In [ ]:
from __future__ import annotations

import os
import random
import subprocess
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import ImageFile
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

# input flag
# "simple_cnn" or "alexnet_scratch"
ARCHITECTURE = "simple_cnn" 

SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 8
EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.0
EARLY_STOPPING_PATIENCE = 10

DATASET_HANDLE = "nocturnalnerd/balanced-dataset-701020"
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = WORK_DIR / f"best_{ARCHITECTURE}.pt"
LATEST_IN_PROGRESS_PATH = WORK_DIR / f"latest_{ARCHITECTURE}_in_progress.pt"
EPOCH_CHECKPOINT_DIR = WORK_DIR / f"epoch_checkpoints_{ARCHITECTURE}"
EPOCH_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


def resolve_dataset_root() -> Path:
    explicit_root = os.environ.get("DATASET_ROOT")
    if explicit_root:
        explicit_path = Path(explicit_root)
        if explicit_path.exists():
            return explicit_path
        raise FileNotFoundError(f"DATASET_ROOT={explicit_path} does not exist.")
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub
    downloaded_root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    direct = downloaded_root / "dataset_result_70_10_20_balanced"
    if direct.exists():
        return direct
    discovered = [p for p in downloaded_root.rglob("dataset_result_70_10_20_balanced") if p.is_dir()]
    if len(discovered) == 1:
        return discovered[0]
    if all((downloaded_root / split).is_dir() for split in ("train", "validation", "test")):
        return downloaded_root
    raise FileNotFoundError(f"Could not locate dataset_result_balanced under {downloaded_root}.")


seed_everything()
DATASET_ROOT = resolve_dataset_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset:", DATASET_ROOT)
print("Architecture:", ARCHITECTURE)
print("Checkpoint target:", CHECKPOINT_PATH)


class FakePositiveImageFolder(ImageFolder):
    """ImageFolder that always returns real=0 and fake=1 plus the image path."""

    def __init__(self, root: Path, transform=None):
        super().__init__(str(root), transform=transform)
        if set(self.classes) != {"fake", "real"}:
            raise ValueError(f"Expected class folders {{'real', 'fake'}} at {root}; found {self.classes}.")

    def __getitem__(self, index: int):
        image, original_target = super().__getitem__(index)
        path, _ = self.samples[index]
        class_name = self.classes[original_target]
        label = torch.tensor(float(class_name == "fake"), dtype=torch.float32)
        return image, label, path


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

train_dataset = FakePositiveImageFolder(DATASET_ROOT / "train", train_transform)
val_dataset = FakePositiveImageFolder(DATASET_ROOT / "validation", eval_transform)


def class_counts(dataset: FakePositiveImageFolder) -> dict[str, int]:
    counts = Counter(dataset.classes[target] for _, target in dataset.samples)
    return {name: counts[name] for name in ("real", "fake")}


for split_name, split_dataset in (("train", train_dataset), ("validation", val_dataset)):
    print(f"{split_name:10s} {class_counts(split_dataset)}")

loader_generator = torch.Generator().manual_seed(SEED)
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None, generator=loader_generator,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=pin_memory, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)



class SimpleCNN(nn.Module):
    """A shallow, non-residual, purely spatial-domain CNN classifier.

    Five conv blocks (channel progression 32-64-128-256-256), each with
    BatchNorm + ReLU, four of them followed by 2x max pooling; global average
    pooling replaces a final max pool so the network accepts any input
    resolution. A single linear layer maps directly to one logit -- no
    frequency-domain processing anywhere in the pipeline. This is the
    "generic spatial-domain detector" class of baseline discussed in
    Section II.A: intentionally shallow and residual-free, in contrast to
    both FreqNet's backbone and AlexNet's heavier three-layer classifier
    head, so it occupies a clearly distinct point in the architecture
    comparison rather than duplicating either.
    """

    def __init__(self):
        super().__init__()

        def block(in_ch: int, out_ch: int, pool: bool = True) -> nn.Sequential:
            layers = [nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            if pool:
                layers.append(nn.MaxPool2d(2))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            block(3, 32),
            block(32, 64),
            block(64, 128),
            block(128, 256),
            block(256, 256, pool=False),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, 1)

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


class AlexNetScratch(nn.Module):
    """Krizhevsky et al. (2012) AlexNet, trained from scratch.

    Kernel sizes, strides, padding, and channel progression follow the
    original paper's five conv layers and three FC layers (including 0.5
    dropout on FC6/FC7, preserved because the classifier head is large
    relative to this dataset and remains a real overfitting risk). Two
    disclosed deviations from the original design -- BatchNorm after every
    conv, and AdaptiveAvgPool2d(6, 6) replacing the fixed-size flatten -- are
    explained in the module docstring above.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2), nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(96, 256, kernel_size=5, padding=2), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(256, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096), nn.ReLU(inplace=True),
            nn.Linear(4096, 1),
        )

        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.classifier(x).flatten()


def build_model(architecture: str) -> nn.Module:
    if architecture == "simple_cnn":
        return SimpleCNN()
    if architecture == "alexnet_scratch":
        return AlexNetScratch()
    raise ValueError(f"Unknown ARCHITECTURE: {architecture!r}")


model = build_model(ARCHITECTURE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Architecture: {ARCHITECTURE}  |  Trainable parameters: {n_params:,}")

print("GPU count:", torch.cuda.device_count())
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
use_grad_scaler = use_amp and amp_dtype == torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=use_grad_scaler, init_scale=1024.0, growth_interval=2000)
print("AMP enabled:", use_amp, "| dtype:", amp_dtype if use_amp else "FP32")


def run_training_epoch(loader: DataLoader) -> float:
    model.train()
    running_loss, seen = 0.0, 0
    progress = tqdm(loader, desc="train", leave=False)

    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
            logits = model(images)
        loss = criterion(logits.float(), labels.float())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        progress.set_postfix(loss=f"{running_loss / seen:.4f}")

    return running_loss / seen


@torch.inference_mode()
def predict(loader: DataLoader):
    model.eval()
    running_loss, seen = 0.0, 0
    all_labels, all_probabilities = [], []
    progress = tqdm(loader, desc="evaluate", leave=False)

    for images, labels, _ in progress:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits.float(), labels.float())
        probabilities = torch.sigmoid(logits.float())

        batch_size = labels.numel()
        running_loss += loss.item() * batch_size
        seen += batch_size
        all_labels.append(labels.detach().cpu().numpy().astype(np.int64))
        all_probabilities.append(probabilities.detach().cpu().numpy())

    return running_loss / seen, np.concatenate(all_labels), np.concatenate(all_probabilities)




start_epoch = 1
best_val_auc = -np.inf
epochs_without_improvement = 0
history: list[dict[str, float]] = []

#resume mechanism
if LATEST_IN_PROGRESS_PATH.exists():
    print(f"Found an in-progress checkpoint at {LATEST_IN_PROGRESS_PATH}, resuming.")
    checkpoint = torch.load(LATEST_IN_PROGRESS_PATH, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch = checkpoint["epoch"] + 1
    best_val_auc = checkpoint["best_val_roc_auc_so_far"]
    print(f"Resuming from epoch {start_epoch}, best val ROC-AUC so far: {best_val_auc:.5f}")

for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\n{'=' * 70}\nEpoch {epoch}/{EPOCHS} [{ARCHITECTURE}]\n{'=' * 70}")

    train_loss = run_training_epoch(train_loader)
    val_loss, val_y, val_probability = predict(val_loader)
    val_auc = roc_auc_score(val_y, val_probability)
    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(
        {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_roc_auc": val_auc, "learning_rate": current_lr}
    )
    print(f"Train Loss : {train_loss:.6f}\nVal Loss   : {val_loss:.6f}\nVal ROC-AUC: {val_auc:.6f}\nLR         : {current_lr:.8f}")

    checkpoint_payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_roc_auc": val_auc,
        "learning_rate": current_lr,
        "architecture": ARCHITECTURE,
        "image_size": IMAGE_SIZE,
        "n_params": n_params,
        "label_mapping": {"real": 0, "fake": 1},
        "best_val_roc_auc_so_far": max(best_val_auc, val_auc),
    }
    torch.save(checkpoint_payload, LATEST_IN_PROGRESS_PATH)
    torch.save(checkpoint_payload, EPOCH_CHECKPOINT_DIR / f"model_epoch_{epoch:03d}.pt")

    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0
        torch.save({**checkpoint_payload, "best_val_roc_auc": best_val_auc}, CHECKPOINT_PATH)
        print(f"Saved NEW BEST checkpoint: validation ROC-AUC={best_val_auc:.5f}")
    else:
        epochs_without_improvement += 1
        print(f"No ROC-AUC improvement ({epochs_without_improvement}/{EARLY_STOPPING_PATIENCE})")

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}.\nBest validation ROC-AUC: {best_val_auc:.5f}")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(WORK_DIR / f"history_{ARCHITECTURE}.csv", index=False)
print(history_df)
print(f"\nTraining completed. Best validation ROC-AUC: {best_val_auc:.5f}")
print(f"Best {ARCHITECTURE} checkpoint: {CHECKPOINT_PATH}")

Device: cuda
Dataset: /kaggle/input/datasets/nocturnalnerd/balanced-dataset-701020/dataset_result_70_10_20_balanced
Architecture: simple_cnn
Checkpoint target: /kaggle/working/best_simple_cnn.pt
train      {'real': 19368, 'fake': 19368}
validation {'real': 2767, 'fake': 2767}
Architecture: simple_cnn  |  Trainable parameters: 980,225
GPU count: 2
Using 2 GPUs


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


AMP enabled: True | dtype: torch.bfloat16

Epoch 1/30 [simple_cnn]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


train:   0%|          | 0/2421 [00:00<?, ?it/s]

evaluate:   0%|          | 0/346 [00:00<?, ?it/s]

... (84 outputs left out)

Train Loss : 0.306729
Val Loss   : 0.292848
Val ROC-AUC: 0.949991
LR         : 0.00001250
Saved NEW BEST checkpoint: validation ROC-AUC=0.94999

Epoch 30/30 [simple_cnn]


train:   0%|          | 0/2421 [00:00<?, ?it/s]

evaluate:   0%|          | 0/346 [00:00<?, ?it/s]

Train Loss : 0.302689
Val Loss   : 0.320659
Val ROC-AUC: 0.945618
LR         : 0.00001250
No ROC-AUC improvement (1/10)
    epoch  train_loss  val_loss  val_roc_auc  learning_rate
0       1    0.594783  0.581702     0.787994       0.000100
1       2    0.542479  0.526787     0.833226       0.000100
2       3    0.517673  0.470565     0.857733       0.000100
3       4    0.497871  0.481206     0.863992       0.000100
4       5    0.485261  0.456664     0.871738       0.000100
5       6    0.469691  0.464592     0.872228       0.000100
6       7    0.460632  0.421551     0.889032       0.000100
7       8    0.454068  0.439016     0.880555       0.000100
8       9    0.440939  0.480701     0.883051       0.000100
9      10    0.435177  0.469012     0.875308       0.000050
10     11    0.406598  0.377919     0.914898       0.000050
11     12    0.396629  0.386179     0.913138       0.000050
12     13    0.389775  0.400855     0.902800       0.000050
13     14    0.388068  0.350942     0.92